# Which features earn their place

Day 7's logistic regression (C = 0.1) is refitted on the training seasons with each feature set and scored on validation against **plain Elo** (log loss 0.6411). One more candidate keeps Elo's log-odds exactly as they are and only learns a shift for back-to-backs. Run by `make ablation`.

In [1]:
import json
from pathlib import Path

import pandas as pd

report = json.loads(Path("../reports/ablation.json").read_text())
table = pd.DataFrame(report["sets"]).T
table[["log_loss", "minus_plain_elo_95", "accuracy", "mean_predicted"]].sort_values("log_loss")

,log_loss,minus_plain_elo_95,accuracy,mean_predicted
plain Elo + back-to-back adjustment,0.6393,"[-0.0038, 0.0001]",0.6355,0.5798
Elo + rest,0.6396,"[-0.0038, 0.0006]",0.6368,0.5787
everything minus form,0.6396,"[-0.0038, 0.0006]",0.6368,0.5787
everything,0.6418,"[-0.0033, 0.0044]",0.6342,0.5779
Elo only,0.6425,"[0.0006, 0.0023]",0.6342,0.5827
Elo + form,0.6448,"[0.0003, 0.0069]",0.6312,0.582
everything minus rest,0.6448,"[0.0003, 0.0069]",0.6312,0.582
"form + rest, no Elo",0.6578,"[0.0063, 0.0265]",0.6277,0.5762
everything minus elo,0.6578,"[0.0063, 0.0265]",0.6277,0.5762


In [2]:
adjustment = report["sets"]["plain Elo + back-to-back adjustment"]["coefficients"]
{
    "adjustment (log-odds)": adjustment,
    "rest added to the Elo regression, 95%": report["elo_plus_rest_minus_elo_only_95"],
}

{'adjustment (log-odds)': {'intercept': -0.0133,
  'home_b2b': -0.1388,
  'away_b2b': 0.2304},
 'rest added to the Elo regression, 95%': [-0.0052, -0.0009]}

## Findings

1. **Without Elo, everything is clearly worse** (0.658 against 0.641; the interval is +0.006 to +0.027). The form features are a weaker copy of what the rating already knows.
2. **The form features hurt once Elo is in.** Elo + form is worse than Elo alone, and dropping form from "everything" improves it. Day 4's season-to-date stats are redundant next to a rating that carries over between seasons.
3. **Refitting Elo inside a regression makes it slightly worse** (+0.0006 to +0.0023). Fitting its slope on 2014-20 bakes in the old home edge a second time.
4. **Rest is the one group that adds information.** Added to the same regression, it lowers log loss by 0.0009 to 0.0052 against Elo alone in that regression. As an adjustment on top of plain Elo, the learned shifts are −0.14 log-odds when the home team is on a back-to-back and +0.23 when the away team is: about −3.5 and +5.7 points for an even game.
5. **But against plain Elo it falls just short of decision 11's bar.** The best set (0.6393) has an interval of −0.0038 to +0.0001: almost certainly better, but not clear of zero.

## Decision (12)

The rule was written before these numbers, so it stands. **The CLI uses plain Elo.** The back-to-back adjustment is **declared now, before the test set is opened, as the one secondary model** reported on day 10, next to plain Elo. The test set (3,690 games) can then confirm or reject it without being used to choose it.